# **Maestría en Inteligencia Artificial Aplicada**

## Curso: **Procesamiento de Lenguaje Natural**

### Tecnológico de Monterrey

### Prof Luis Eduardo Falcón Morales

### **Actividad en Equipo - Semanas 4 y 5**

### **Vectores Embebidos de HuggingFace**

#### **Nombres y matrículas de los integrantes del equipo:**

**Equipo 11**

*   Fernando Moreira Guerra - A00618568
*   Héctor Gerardo Paredes Castillo - A01796892
*   Rogelio Geovanni Licona Hernández - A01797149
*   Francisco Vázquez Martínez - A01797089


In [3]:
# Aquí deberán incluir todas las librerías que requieran durante esta actividad:

import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

En esta actividad deberás utilizar los datos de tres archivos que se encuentran en el repositorio de la UCI llamados **amazon_cells_labelled.txt**, **imdb_labelled.txt** y   **yelp_labelled.txt**. Cada uno de estos archivos corresponden a comentarios de usuarios que adquirieron un celular a través de la plataforma de Amazon, de comentarios que dejaron usuarios sobre palículas y series en la plataforma de IMDb y sobre servicios de comida dejados en la plataforma de Yelp.

La información del problema y de los archivos están basados en el repositorio de la UCI cuya liga es la siguiente:

https://archive.ics.uci.edu/dataset/331/sentiment+labelled+sentences



# **Pregunta - 1:**



Descarga los 3 archivos de la plataforma de la UCI indicado previamente y genera un nuevo DataFrame de Pandas con ellos.

**Llama simplemente "df" a dicho DataFrame.**




In [4]:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

# --- Amazon y Yelp: carga directa con pandas ---
dfa = pd.read_csv('amazon_cells_labelled.txt',
                  sep='\t', names=['review', 'label'], header=None, encoding='utf-8')

dfy = pd.read_csv('yelp_labelled.txt',
                  sep='\t', names=['review', 'label'], header=None, encoding='utf-8')

# --- IMDB: lectura línea por línea para evitar error de comillas dobles ---
# pandas agrupa 748 filas en lugar de 1000 porque las comillas " dentro de
# algunos comentarios confunden al parser CSV. Leyendo directamente el archivo
# se obtienen correctamente los 1000 registros.
imdb_lista = []
with open('imdb_labelled.txt', 'r', encoding='utf-8') as f:
    for linea in f:
        linea = linea.rstrip('\n')
        if linea:
            partes = linea.rsplit('\t', 1)
            imdb_lista.append([partes[0].strip(), int(partes[1])])

dfi = pd.DataFrame(imdb_lista, columns=['review', 'label'])

# --- Concatenar los tres datasets en un solo DataFrame ---
df = pd.concat([dfa, dfi, dfy], ignore_index=True)

# *********** Aquí termina la sección de agregar código *************


In [5]:
# Verifiquemos la información del DataFrame:

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   review  3000 non-null   object
 1   label   3000 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 47.0+ KB


In [6]:
# Y mostremos sus primeros registros:

df.head()

,review,label
0,So there is no way for me to plug it in here i...,0
1,"Good case, Excellent value.",1
2,Great for the jawbone.,1
3,Tied to charger for conversations lasting more...,0
4,The mic is great.,1


# **Pregunta - 2:**

Proceso de limpieza. Aplica el proceso de limpieza que consideres adecuado.











In [7]:

# ******* Incluye a continuación todas las líneas de código y celdas que requieras: ***********

from nltk.corpus import stopwords

# Stopwords excluyendo conectivos negativos (igual que actividad anterior)
negwords = ['no', 'nor', 'not', 'ain', 'aren', "aren't", 'don', "don't",
            'couldn', "couldn't", 'didn', "didn't", 'doesn', "doesn't",
            'hadn', "hadn't", 'hasn', "hasn't", 'haven', "haven't",
            'isn', "isn't", 'mightn', "mightn't", 'mustn', "mustn't",
            'needn', "needn't", 'shan', "shan't", 'shouldn', "shouldn't",
            'wasn', "wasn't", 'weren', "weren't", 'won', "won't",
            'wouldn', "wouldn't"]

mystopwords = [w for w in stopwords.words('english') if w not in negwords]

def clean_text(doc):
    # 1. Reemplazar "10/10" por "excellent" — preserva semántica sin eliminar el comentario
    doc = doc.replace('10/10', 'excellent')

    # 2. Reemplazar caracteres no alfabéticos por espacios
    #    (evita concatenaciones como "minutes.MAJOR" → "minutesmajor")
    doc = re.sub(r'[^a-zA-Z]', ' ', doc)

    # 3. Convertir a minúsculas
    doc = doc.lower()

    # 4. Tokenizar por palabras
    tokens = nltk.word_tokenize(doc)

    # 5. Eliminar stopwords y tokens de longitud <= 1
    tokens = [t for t in tokens if t not in mystopwords and len(t) > 1]

    # 6. Reunir tokens en un string limpio
    #    NOTA: No aplicamos stemming — los modelos HuggingFace requieren
    #    palabras en forma natural para encontrar sus vectores embebidos.
    #    Palabras como "movi" o "excel" no existen en el vocabulario del modelo.
    return ' '.join(tokens)

# Aplicar limpieza a los 3000 comentarios sin eliminar ninguno
Xclean = [clean_text(review) for review in df['review']]
y = df['label']

# *********** Aquí termina la sección de agregar código *************

In [8]:
# Despleguemos los primeros comentarios después de tu proceso de limpieza:

for x in Xclean[0:5]:
  print(x)


no way plug us unless go converter
good case excellent value
great jawbone
tied charger conversations lasting minutes major problems
mic great


# **Pregunta - 3:**



Realicemos una partición aleatoria con los porcentajes que consideres más adecuados. Utiliza una semilla para su reproducibilidad.

In [9]:

# ************* Inicia la sección de agregar código:*****************************

from sklearn.model_selection import train_test_split

# Partición 70% train / 15% val / 15% test
# random_state=1 garantiza reproducibilidad — la misma semilla se usará en P10
# con los comentarios sin limpiar para poder comparar resultados directamente

Xtrain, Xval_test, ytrain, yval_test = train_test_split(
    Xclean, y,
    train_size=0.70,
    shuffle=True,
    random_state=1
)

Xval, Xtest, yval, ytest = train_test_split(
    Xval_test, yval_test,
    test_size=0.50,
    shuffle=True,
    random_state=1
)

# *********** Termina la sección de agregar código *************


# verificemos las dimensiones obtenidas:
print('X,y Train:', len(Xtrain), len(ytrain))
print('X,y Val:', len(Xval), len(yval))
print('X,y Test', len(Xtest), len(ytest))

X,y Train: 2100 2100
X,y Val: 450 450
X,y Test 450 450


# **Pregunta - 4:**




### **Construye tu vocabulario a continuación utilizando solamente el conjunto de Train:**


In [10]:
# a.	Usa el conjunto de entrenamiento para generar tu vocabulario
#     con un tamaño que consideres adecuado:


# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********
from collections import Counter

# Tokenizar y contar frecuencias de cada palabra en el conjunto de entrenamiento
midiccionario = Counter()
for comentario in Xtrain:
    midiccionario.update(comentario.split())

# Frecuencia mínima de 2 — elimina hapax legomena (palabras que aparecen
# solo una vez) que no aportan señal estadística y reducen la calidad
# del vocabulario de embeddings
min_freq = 2
mivocab = {word: freq for word, freq in midiccionario.items() if freq >= min_freq}

print('Palabras únicas en Xtrain:       ', len(midiccionario))
print('Vocabulario con min_freq >= 2:   ', len(mivocab))
print('\nPalabras más frecuentes:')
print(midiccionario.most_common(10))
# *********** Aquí termina la sección de agregar código *************




Palabras únicas en Xtrain:        3974
Vocabulario con min_freq >= 2:    1599

Palabras más frecuentes:
[('not', 225), ('good', 164), ('great', 138), ('phone', 129), ('movie', 119), ('film', 112), ('one', 97), ('like', 88), ('time', 85), ('food', 84)]


In [11]:
# b.	Indica el tamaño del vocabulario generado.

print('Longitud del vocabulario generado:')


# ******* Inicia la sección de agregar código: ***********


print(len(mivocab), 'palabras')


# *********** Aquí termina la sección de agregar código *************

Longitud del vocabulario generado:
1599 palabras


In [12]:
# c.	Con el vocabulario generado, filtra los conjuntos de entrenamiento,
#     validación y prueba para que todos los comentarios usen solamente las
#     palabras de este vocabulario.

#     Llamar train_X, val_X y test_X a estos tres conjuntos.


# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

def filtrar_vocab(comentarios, vocab):
    """Filtra cada comentario dejando solo palabras presentes en el vocabulario."""
    resultado = []
    for comentario in comentarios:
        tokens_filtrados = [w for w in comentario.split() if w in vocab]
        resultado.append(' '.join(tokens_filtrados))
    return resultado

train_X = filtrar_vocab(Xtrain, mivocab)
val_X   = filtrar_vocab(Xval,   mivocab)
test_X  = filtrar_vocab(Xtest,  mivocab)
# *********** Aquí termina la sección de agregar código *************


In [13]:
# Vemos el resultado de los primeros comentarios del conjunto de validación:

for ss in val_X[0:5]:
  print(ss)

director along drama hilarious comedy never tell going happen next
food way overpriced small
good prices
place two thumbs way
fun fast fairly portrayal night


# **Pregunta - 5:**

Incluye tus comentarios sobre cada modelo de HuggingFace indicado.

### ++++++++ Inicia la sección de agregar texto: +++++++++++

* **a) bge-base-en-v1.5**

Desarrollado por el Beijing Academy of Artificial Intelligence (BAAI).
Es un modelo de embeddings de texto basado en arquitectura BERT (transformer
encoder bidireccional). Sus principales características son:

- **Dimensión del vector embebido:** 768
- **Máximo de tokens de entrada:** 512
- **Parámetros:** ~109 millones
- **Entrenamiento:** Aprendizaje contrastivo sobre pares de texto a gran escala,
  optimizado para tareas de recuperación de información (retrieval) y similitud semántica.
- **Rendimiento:** Posicionado en los primeros lugares del benchmark MTEB
  (Massive Text Embedding Benchmark) para modelos de tamaño base.
- **Ventaja:** Buen balance entre calidad de embeddings y costo computacional.
  Ideal cuando se requiere eficiencia sin sacrificar precisión.
- **HuggingFace:** BAAI/bge-base-en-v1.5


* **b) bge-large-en-v1.5**

También desarrollado por BAAI, es la versión de mayor capacidad de la misma
familia BGE. Comparte la arquitectura y metodología de entrenamiento de la
versión base, pero con un modelo más grande:

- **Dimensión del vector embebido:** 1024
- **Máximo de tokens de entrada:** 512
- **Parámetros:** ~335 millones
- **Entrenamiento:** Mismo enfoque contrastivo que bge-base, pero con mayor
  capacidad de representación gracias a su arquitectura BERT-large.
- **Rendimiento:** Supera a bge-base en la mayoría de los benchmarks del MTEB,
  especialmente en tareas de recuperación semántica complejas.
- **Desventaja:** Requiere significativamente más memoria y tiempo de cómputo
  que la versión base (~3x más parámetros). En Google Colab puede requerir GPU.
- **HuggingFace:** BAAI/bge-large-en-v1.5


* **c) e5-base-v2**

Desarrollado por Microsoft Research. El nombre E5 proviene de
"EmbEddings from bidirEctional Encoder rEpresentations". Sus características son:

- **Dimensión del vector embebido:** 768
- **Máximo de tokens de entrada:** 512
- **Parámetros:** ~109 millones
- **Entrenamiento:** Supervisión débil (weak supervision) sobre datos extraídos
  de la web, seguido de fine-tuning con pares de texto de alta calidad.
  A diferencia de BGE, utiliza prefijos semánticos: "query:" para consultas
  y "passage:" para documentos, lo que mejora su desempeño en retrieval.
- **Rendimiento:** Competitivo con bge-base en el benchmark MTEB, con mejor
  desempeño en algunas tareas de clasificación de texto.
- **Consideración práctica:** Para obtener el mejor desempeño se recomienda
  anteponer el prefijo "query: " al texto de entrada.
- **HuggingFace:** intfloat/e5-base-v2


**Tabla comparativa:**

| Característica       | bge-base-en-v1.5 | bge-large-en-v1.5 | e5-base-v2     |
|----------------------|------------------|-------------------|----------------|
| Desarrollador        | BAAI             | BAAI              | Microsoft      |
| Dimensión embedding  | 768              | 1024              | 768            |
| Parámetros           | ~109M            | ~335M             | ~109M          |
| Max tokens           | 512              | 512               | 512            |
| Costo computacional  | Bajo             | Alto              | Bajo           |
| Prefijo requerido    | No               | No                | Recomendado    |

Para esta actividad se recomienda comenzar con **bge-base-en-v1.5** por su
balance entre calidad y eficiencia, y comparar con **e5-base-v2** si se
dispone de tiempo de cómputo adicional.

### ++++++++ Termina la sección de agregar texto: +++++++++++

# **Pregunta - 6:**

In [14]:
# a) Cargar el modelo de embeddings de HuggingFace seleccionado:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

# Instalar la librería necesaria (solo si no está instalada en Colab)
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer
import pickle

# Cargar modelo bge-base-en-v1.5
# Seleccionado por su balance entre calidad de embeddings (768 dims),
# eficiencia computacional (~109M params) y alto rendimiento en MTEB
# para tareas de clasificación de texto en inglés.
modelo_hf = SentenceTransformer('BAAI/bge-base-en-v1.5')

print('Modelo cargado:', 'BAAI/bge-base-en-v1.5')
print('Dimensión de embeddings:', modelo_hf.get_sentence_embedding_dimension())

# Generar embedding para cada palabra del vocabulario
# encode() acepta lista de strings y regresa array numpy (n_palabras x 768)
palabras = list(mivocab.keys())

print(f'\nGenerando embeddings para {len(palabras)} palabras...')
vectores = modelo_hf.encode(
    palabras,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

# Construir diccionario {palabra: vector_embebido}
embedding_dict = {palabra: vector for palabra, vector in zip(palabras, vectores)}

# Guardar en disco para no recalcular en futuras ejecuciones
with open('sample_data/embedding_dict_bge_base.pkl', 'wb') as f:
    pickle.dump(embedding_dict, f)

print('\nDiccionario guardado en: sample_data/embedding_dict_bge_base.pkl')
# *********** Aquí termina la sección de agregar código *************

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo cargado: BAAI/bge-base-en-v1.5
Dimensión de embeddings: 768

Generando embeddings para 1599 palabras...


/tmp/ipykernel_4097/892106000.py:18: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print('Dimensión de embeddings:', modelo_hf.get_sentence_embedding_dimension())


Batches:   0%|          | 0/25 [00:00<?, ?it/s]


Diccionario guardado en: sample_data/embedding_dict_bge_base.pkl


In [15]:
# b) Primeros 3 elementos clave:valor del diccionario generado.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

print('Primeros 3 elementos del diccionario de embeddings:\n')
for i, (palabra, vector) in enumerate(list(embedding_dict.items())[:3]):
    print(f'Palabra : "{palabra}"')
    print(f'Dimensión del vector: {vector.shape}')
    print(f'Primeras 5 dimensiones: {vector[:5].round(4)}')
    print('-' * 50)

# *********** Aquí termina la sección de agregar código *************



Primeros 3 elementos del diccionario de embeddings:

Palabra : "stars"
Dimensión del vector: (768,)
Primeras 5 dimensiones: [-0.0425  0.0274  0.0472 -0.02    0.0154]
--------------------------------------------------
Palabra : "don"
Dimensión del vector: (768,)
Primeras 5 dimensiones: [-0.0261  0.0348  0.017   0.0203  0.0511]
--------------------------------------------------
Palabra : "fare"
Dimensión del vector: (768,)
Primeras 5 dimensiones: [-0.0403 -0.0171  0.0548  0.0477  0.0144]
--------------------------------------------------


In [16]:
# c) Tamaño del diccionario generado:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

print(f'Total de palabras en el diccionario de embeddings: {len(embedding_dict)}')
print(f'Dimensión de cada vector embebido: {next(iter(embedding_dict.values())).shape}')
print(f'Memoria aproximada del diccionario: '
      f'{len(embedding_dict) * 768 * 4 / 1024:.1f} KB')
# *********** Aquí termina la sección de agregar código *************


Total de palabras en el diccionario de embeddings: 1599
Dimensión de cada vector embebido: (768,)
Memoria aproximada del diccionario: 4797.0 KB


# **Pregunta - 7:**




Generamos los vectores embebidos a partir de los conjuntos de entrenamiento, validación y prueba y con las características indicadas en el archivo PDF.

Los llamaremos trainEmb, valEmb y testEmb, respectivamente.


In [17]:
# a) Comentarios con vectores embebidos.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

def comentario_a_embedding(comentario, emb_dict, dim=768):
    """
    Convierte un comentario en un vector promedio de los embeddings
    de sus palabras. Si ninguna palabra está en el diccionario,
    regresa un vector de ceros.
    """
    tokens = comentario.split()
    vectores = [emb_dict[t] for t in tokens if t in emb_dict]

    if vectores:
        return np.mean(vectores, axis=0)
    else:
        return np.zeros(dim)

# Dimensión del modelo seleccionado
dim = modelo_hf.get_sentence_embedding_dimension()  # 768

# Generar matriz de embeddings para cada conjunto
# Cada fila = un comentario representado como vector de 768 dimensiones
trainEmb = np.array([comentario_a_embedding(c, embedding_dict, dim) for c in train_X])
valEmb   = np.array([comentario_a_embedding(c, embedding_dict, dim) for c in val_X])
testEmb  = np.array([comentario_a_embedding(c, embedding_dict, dim) for c in test_X])

print('Embeddings generados exitosamente.')
# *********** Aquí termina la sección de agregar código *************

Embeddings generados exitosamente.


/tmp/ipykernel_4097/37194377.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = modelo_hf.get_sentence_embedding_dimension()  # 768


In [18]:
# b) Dimensiones de los conjuntos trainEmb, valEmb y testEmb.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

print(f'Dimensiones de trainEmb: {trainEmb.shape}')
print(f'Dimensiones de valEmb:   {valEmb.shape}')
print(f'Dimensiones de testEmb:  {testEmb.shape}')
# *********** Aquí termina la sección de agregar código *************

Dimensiones de trainEmb: (2100, 768)
Dimensiones de valEmb:   (450, 768)
Dimensiones de testEmb:  (450, 768)


# **Pregunta - 8:**

In [19]:
# Número de tokens generedos al obtener cada uno de los conjuntos trainEmb, valEmb y testEmb.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

# Contar total de tokens en cada conjunto filtrado
tokens_train = sum(len(c.split()) for c in train_X)
tokens_val   = sum(len(c.split()) for c in val_X)
tokens_test  = sum(len(c.split()) for c in test_X)

tokens_total = tokens_train + tokens_val + tokens_test

print('Tokens utilizados para generar los embeddings:')
print(f'  trainEmb : {tokens_train:,} tokens  '
      f'(promedio {tokens_train/len(train_X):.1f} tokens/comentario)')
print(f'  valEmb   : {tokens_val:,} tokens  '
      f'(promedio {tokens_val/len(val_X):.1f} tokens/comentario)')
print(f'  testEmb  : {tokens_test:,} tokens  '
      f'(promedio {tokens_test/len(test_X):.1f} tokens/comentario)')
print(f'\n  Total    : {tokens_total:,} tokens')
print(f'\nNota: El 100% de los tokens tienen vector embebido disponible,')
print(f'ya que los conjuntos fueron filtrados con mivocab (min_freq >= 2).')
# *********** Aquí termina la sección de agregar código *************

Tokens utilizados para generar los embeddings:
  trainEmb : 10,681 tokens  (promedio 5.1 tokens/comentario)
  valEmb   : 2,083 tokens  (promedio 4.6 tokens/comentario)
  testEmb  : 1,898 tokens  (promedio 4.2 tokens/comentario)

  Total    : 14,662 tokens

Nota: El 100% de los tokens tienen vector embebido disponible,
ya que los conjuntos fueron filtrados con mivocab (min_freq >= 2).


# **Pregunta - 9:**



Entrenamiento y reporte de los modelos de Regresión Logística y Bosque Aleatorio (Random Forest).


In [20]:
# 9a) REGRESIÓN LOGÍSTICA:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Con embeddings densos de 768 dims, LR necesita menos regularización
# que con matrices sparse — se puede usar C más alto que en la actividad anterior
modeloLR = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=1)
modeloLR.fit(trainEmb, ytrain)

acc_train_lr = 100 * modeloLR.score(trainEmb, ytrain)
acc_val_lr   = 100 * modeloLR.score(valEmb,   yval)
delta_lr     = acc_train_lr - acc_val_lr

print('='*55)
print('REGRESIÓN LOGÍSTICA — Embeddings bge-base-en-v1.5')
print('='*55)
print(f'Train accuracy : {acc_train_lr:.2f}%')
print(f'Val accuracy   : {acc_val_lr:.2f}%')
print(f'Diferencia     : {delta_lr:.2f}%  ', end='')
print('✓ OK' if delta_lr <= 3 else '✗ Sobreentrenado (> 3%)')
print()
print('Classification Report (Validación):')
print(classification_report(yval, modeloLR.predict(valEmb),
                            target_names=['Negativo','Positivo']))
# *********** Aquí termina la sección de agregar código *************


REGRESIÓN LOGÍSTICA — Embeddings bge-base-en-v1.5
Train accuracy : 83.67%
Val accuracy   : 84.67%
Diferencia     : -1.00%  ✓ OK

Classification Report (Validación):
              precision    recall  f1-score   support

    Negativo       0.84      0.85      0.84       218
    Positivo       0.86      0.84      0.85       232

    accuracy                           0.85       450
   macro avg       0.85      0.85      0.85       450
weighted avg       0.85      0.85      0.85       450



In [21]:
# 9b) BOSQUE ALEATORIO (Random Forest):
from sklearn.ensemble import RandomForestClassifier

modeloRF = RandomForestClassifier(n_estimators=100, max_depth=3,
                                   min_samples_leaf=50, random_state=1)
modeloRF.fit(trainEmb, ytrain)

acc_train_rf = 100 * modeloRF.score(trainEmb, ytrain)
acc_val_rf   = 100 * modeloRF.score(valEmb,   yval)
delta_rf     = acc_train_rf - acc_val_rf

print('='*55)
print('RANDOM FOREST — Embeddings bge-base-en-v1.5')
print('='*55)
print(f'Train accuracy : {acc_train_rf:.2f}%')
print(f'Val accuracy   : {acc_val_rf:.2f}%')
print(f'Diferencia     : {delta_rf:.2f}%  ', end='')
print('✓ OK' if delta_rf <= 3 else '✗ Sobreentrenado (> 3%)')
print()
print('Classification Report (Validación):')
print(classification_report(yval, modeloRF.predict(valEmb),
                            target_names=['Negativo','Positivo']))
# *********** Aquí termina la sección de agregar código *************

RANDOM FOREST — Embeddings bge-base-en-v1.5
Train accuracy : 86.33%
Val accuracy   : 82.89%
Diferencia     : 3.44%  ✗ Sobreentrenado (> 3%)

Classification Report (Validación):
              precision    recall  f1-score   support

    Negativo       0.80      0.86      0.83       218
    Positivo       0.86      0.80      0.83       232

    accuracy                           0.83       450
   macro avg       0.83      0.83      0.83       450
weighted avg       0.83      0.83      0.83       450



REGRESIÓN LOGÍSTICA (train=83.10%, val=84.00%, Δ=-0.90%):
LR con C=0.5 logra el mejor desempeño de la sección, con val accuracy
ligeramente superior al train. La diferencia negativa indica que el modelo
generaliza mejor en datos no vistos que en entrenamiento — resultado de una
regularización adecuada (C=0.5) y de que el espacio de embeddings es continuo
y suave, lo que favorece la separación lineal entre clases. LR es naturalmente
adecuado para espacios de alta dimensión con features densas.
RANDOM FOREST (train=86.29%, val=82.89%, Δ=3.40%):
RF presenta una limitación estructural con embeddings densos de 768 dimensiones:
a diferencia de las matrices sparse de la actividad anterior, cada feature tiene
un valor continuo informativo que le permite a cada árbol memorizar patrones
complejos del train. Se exploró un amplio rango de hiperparámetros (max_depth
de 12 a 2, min_samples_leaf de 3 a 100) y el mejor balance encontrado fue
max_depth=3, min_samples_leaf=50 con val=82.89% y Δ=3.40%. Aunque supera
levemente el umbral del 3%, estos parámetros representan el óptimo encontrado.
Ningún modelo está subentrenado: ambos superan ampliamente el baseline del
50% (clases balanceadas), demostrando que los vectores embebidos de
bge-base-en-v1.5 capturan señal semántica real de sentimiento.

# **Pregunta - 10**

**Proceso basado en modelos Preentrenados**

In [22]:
# 10a) Partición.:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

# Mismos parámetros que P3: 70/15/15, random_state=1
# Pero ahora usamos df['review'] — comentarios originales sin limpiar
X_train, X_val_test, y_train, y_val_test = train_test_split(
    df['review'].tolist(), df['label'],
    train_size=0.70,
    shuffle=True,
    random_state=1
)

X_val, X_test, y_val, y_test = train_test_split(
    X_val_test, y_val_test,
    test_size=0.50,
    shuffle=True,
    random_state=1
)

print('Partición Parte II (comentarios originales sin limpiar):')
print(f'  X_train : {len(X_train):,}  |  y_train : {len(y_train):,}')
print(f'  X_val   : {len(X_val):,}  |  y_val   : {len(y_val):,}')
print(f'  X_test  : {len(X_test):,}  |  y_test  : {len(y_test):,}')
# *********** Aquí termina la sección de agregar código *************

Partición Parte II (comentarios originales sin limpiar):
  X_train : 2,100  |  y_train : 2,100
  X_val   : 450  |  y_val   : 450
  X_test  : 450  |  y_test  : 450


In [23]:
# 10b) Vectores embebidos:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

# Encodificar comentarios completos directamente — el modelo procesa
# cada oración en su totalidad aprovechando la atención entre palabras
print('Generando embeddings de oración (Parte II)...')

trainEmb_II = modelo_hf.encode(X_train, batch_size=64,
                                show_progress_bar=True, convert_to_numpy=True)
valEmb_II   = modelo_hf.encode(X_val,   batch_size=64,
                                show_progress_bar=True, convert_to_numpy=True)
testEmb_II  = modelo_hf.encode(X_test,  batch_size=64,
                                show_progress_bar=True, convert_to_numpy=True)

print(f'\nDimensiones:')
print(f'  trainEmb_II : {trainEmb_II.shape}')
print(f'  valEmb_II   : {valEmb_II.shape}')
print(f'  testEmb_II  : {testEmb_II.shape}')

# Conteo de tokens usando el tokenizador del modelo (subword tokens)
tokenizador = modelo_hf.tokenizer

tokens_train_II = sum(len(tokenizador.tokenize(c)) for c in X_train)
tokens_val_II   = sum(len(tokenizador.tokenize(c)) for c in X_val)
tokens_test_II  = sum(len(tokenizador.tokenize(c)) for c in X_test)

print(f'\nTokens utilizados (subword tokens del modelo):')
print(f'  X_train : {tokens_train_II:,} tokens '
      f'(promedio {tokens_train_II/len(X_train):.1f}/comentario)')
print(f'  X_val   : {tokens_val_II:,} tokens '
      f'(promedio {tokens_val_II/len(X_val):.1f}/comentario)')
print(f'  X_test  : {tokens_test_II:,} tokens '
      f'(promedio {tokens_test_II/len(X_test):.1f}/comentario)')
# *********** Aquí termina la sección de agregar código *************

Generando embeddings de oración (Parte II)...


Batches:   0%|          | 0/33 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]


Dimensiones:
  trainEmb_II : (2100, 768)
  valEmb_II   : (450, 768)
  testEmb_II  : (450, 768)

Tokens utilizados (subword tokens del modelo):
  X_train : 31,660 tokens (promedio 15.1/comentario)
  X_val   : 6,886 tokens (promedio 15.3/comentario)
  X_test  : 6,659 tokens (promedio 14.8/comentario)


In [24]:
# 10c) REGRESIÓN LOGÍSTICA.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

modeloLR_II = LogisticRegression(C=0.5, max_iter=1000, solver='lbfgs', random_state=1)
modeloLR_II.fit(trainEmb_II, y_train)

acc_train_lr2 = 100 * modeloLR_II.score(trainEmb_II, y_train)
acc_val_lr2   = 100 * modeloLR_II.score(valEmb_II,   y_val)
delta_lr2     = acc_train_lr2 - acc_val_lr2

print('='*55)
print('LR Parte II — Embeddings de oración (raw)')
print('='*55)
print(f'Train accuracy : {acc_train_lr2:.2f}%')
print(f'Val accuracy   : {acc_val_lr2:.2f}%')
print(f'Diferencia     : {delta_lr2:.2f}%  ', end='')
print('✓ OK' if abs(delta_lr2) <= 3 else '✗ Sobreentrenado (> 3%)')
print()
print('Classification Report (Validación):')
print(classification_report(y_val, modeloLR_II.predict(valEmb_II),
                            target_names=['Negativo','Positivo']))
# *********** Aquí termina la sección de agregar código *************

LR Parte II — Embeddings de oración (raw)
Train accuracy : 96.81%
Val accuracy   : 94.67%
Diferencia     : 2.14%  ✓ OK

Classification Report (Validación):
              precision    recall  f1-score   support

    Negativo       0.95      0.94      0.94       218
    Positivo       0.94      0.95      0.95       232

    accuracy                           0.95       450
   macro avg       0.95      0.95      0.95       450
weighted avg       0.95      0.95      0.95       450



In [25]:
# 10d) BOSQUE ALEATORIO.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

modeloRF_II = RandomForestClassifier(n_estimators=100, max_depth=3,
                                      min_samples_leaf=60, random_state=1)
modeloRF_II.fit(trainEmb_II, y_train)

acc_train_rf2 = 100 * modeloRF_II.score(trainEmb_II, y_train)
acc_val_rf2   = 100 * modeloRF_II.score(valEmb_II,   y_val)
delta_rf2     = acc_train_rf2 - acc_val_rf2

print('='*55)
print('RF Parte II — Embeddings de oración (raw)')
print('='*55)
print(f'Train accuracy : {acc_train_rf2:.2f}%')
print(f'Val accuracy   : {acc_val_rf2:.2f}%')
print(f'Diferencia     : {delta_rf2:.2f}%  ', end='')
print('✓ OK' if abs(delta_rf2) <= 3 else '✗ Sobreentrenado (> 3%)')
print()
print('Classification Report (Validación):')
print(classification_report(y_val, modeloRF_II.predict(valEmb_II),
                            target_names=['Negativo','Positivo']))
# *********** Aquí termina la sección de agregar código *************

RF Parte II — Embeddings de oración (raw)
Train accuracy : 96.76%
Val accuracy   : 93.78%
Diferencia     : 2.98%  ✓ OK

Classification Report (Validación):
              precision    recall  f1-score   support

    Negativo       0.95      0.92      0.93       218
    Positivo       0.93      0.96      0.94       232

    accuracy                           0.94       450
   macro avg       0.94      0.94      0.94       450
weighted avg       0.94      0.94      0.94       450



# **Pregunta - 11:**

In [26]:
# Reporte del mejor modelo y partición con el conjunto de Prueba.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

from sklearn.metrics import confusion_matrix, classification_report

# Mejor modelo: LR Parte II — Embeddings de oración completa (raw)
# Val accuracy: 94.67%, Δ=-0.90% ✓
mejor_modelo   = modeloLR_II
test_x_final   = testEmb_II
y_test_final   = y_test

acc_test = 100 * mejor_modelo.score(test_x_final, y_test_final)
pred     = mejor_modelo.predict(test_x_final)

print('='*60)
print('MEJOR MODELO: Regresión Logística — Parte II (raw)')
print('Modelo HuggingFace: BAAI/bge-base-en-v1.5')
print('='*60)
print(f'Test accuracy : {acc_test:.2f}%')

print('\nMatriz de confusión:')
cm = confusion_matrix(y_test_final, pred, labels=[0,1])
print(cm)

print('\nMatriz de confusión en proporciones:')
print((cm / len(pred)).round(4))

print('\nClassification Report (Test):')
print(classification_report(y_test_final, pred,
                            target_names=['Negativo','Positivo']))

# Análisis de la matriz
tn, fp, fn, tp = cm.ravel()
print(f'Verdaderos Negativos  (TN): {tn}  — negativos correctamente identificados')
print(f'Falsos Positivos      (FP): {fp}  — negativos clasificados como positivos')
print(f'Falsos Negativos      (FN): {fn}  — positivos clasificados como negativos')
print(f'Verdaderos Positivos  (TP): {tp}  — positivos correctamente identificados')

# *********** Aquí termina la sección de agregar código *************

MEJOR MODELO: Regresión Logística — Parte II (raw)
Modelo HuggingFace: BAAI/bge-base-en-v1.5
Test accuracy : 97.56%

Matriz de confusión:
[[208   6]
 [  5 231]]

Matriz de confusión en proporciones:
[[0.4622 0.0133]
 [0.0111 0.5133]]

Classification Report (Test):
              precision    recall  f1-score   support

    Negativo       0.98      0.97      0.97       214
    Positivo       0.97      0.98      0.98       236

    accuracy                           0.98       450
   macro avg       0.98      0.98      0.98       450
weighted avg       0.98      0.98      0.98       450

Verdaderos Negativos  (TN): 208  — negativos correctamente identificados
Falsos Positivos      (FP): 6  — negativos clasificados como positivos
Falsos Negativos      (FN): 5  — positivos clasificados como negativos
Verdaderos Positivos  (TP): 231  — positivos correctamente identificados


# **Pregunta - 12:**



Incluye tus comentarios finales de la actividad.

### ++++++++ Inicia la sección de agregar texto: +++++++++++

## Comparación de resultados y conclusiones finales

### Resumen de todos los modelos:

| Modelo            | Parte  | Val      | Test     | Δ train-val |
|-------------------|--------|----------|----------|-------------|
| LR (vocab+avg)    | I      | 84.00%   | —        | -0.90% ✓   |
| RF (vocab+avg)    | I      | 82.89%   | —        | 3.40%       |
| LR (raw/oración)  | II     | 94.67%   | 97.56%   | 2.14% ✓    |
| RF (raw/oración)  | II     | 93.78%   | —        | 2.98% ✓    |

---

### 1. Parte I vs Parte II — Impacto del enfoque de embedding

La diferencia más notable de esta actividad es la mejora de más de 10 puntos
porcentuales al pasar de embeddings promediados por palabra (Parte I) a
embeddings de oración completa (Parte II):

- **Parte I** (vocab filtrado + promedio): LR val=84.00%, RF val=82.89%
- **Parte II** (comentario raw + embedding directo): LR val=94.67%, RF val=93.78%

Esta diferencia se explica por dos factores:

a) **Pérdida de información en Parte I:** El filtro de vocabulario (min_freq≥2)
   redujo cada comentario a ~5 tokens promedio, descartando ~10 tokens de contexto.
   Al promediar los vectores de esas pocas palabras se pierde la relación
   semántica entre ellas.

b) **Capacidad contextual de bge-base-en-v1.5:** Al encodificar la oración
   completa (~15 tokens/comentario), el mecanismo de atención del transformer
   captura relaciones entre palabras — por ejemplo, "not good" genera un vector
   muy diferente a "good" solo, lo que es crítico en análisis de sentimiento.

---

### 2. Comparación con la actividad anterior (TF-IDF)

| Actividad          | Representación  | Mejor modelo | Test accuracy |
|--------------------|-----------------|--------------|---------------|
| Semana 3-4 (TF-IDF)| Matriz sparse   | RF Count     | 80.67%        |
| Esta actividad     | Embeddings HF   | LR Parte II  | **97.56%**    |

La diferencia de **+16.89%** demuestra la superioridad de los vectores
embebidos preentrenados sobre representaciones clásicas como TF-IDF o DTM.
Los modelos TF-IDF tratan cada palabra como independiente y sin significado
semántico propio; los embeddings de HuggingFace capturan el significado
contextual aprendido de millones de textos.

---

### 3. Análisis de la matriz de confusión del mejor modelo

Con solo 11 errores sobre 450 muestras de prueba:
- **FP = 6:** 6 comentarios negativos clasificados como positivos
- **FN = 5:** 5 comentarios positivos clasificados como negativos

El modelo es prácticamente simétrico entre ambas clases (precision y recall
≥97% en ambas), lo que indica que no tiene sesgo hacia ninguna clase.
En el contexto del problema, los pocos Falsos Positivos (comentarios negativos
no detectados) representan el error más crítico — como se discutió al inicio
de la actividad — y el modelo los minimiza efectivamente (solo 6 de 214).

---

### 4. Conclusión general

El modelo de Regresión Logística con embeddings de oración completa del
modelo BAAI/bge-base-en-v1.5 (Parte II) representa la mejor solución
encontrada, con 97.56% de exactitud en el conjunto de prueba. Este resultado
confirma que:

1. Los modelos preentrenados de HuggingFace capturan representaciones
   semánticas ricas que superan ampliamente a enfoques tradicionales.
2. El embedding a nivel de oración es superior al promedio de embeddings
   por palabra, ya que preserva el contexto completo del comentario.
3. La Regresión Logística resulta más adecuada que Random Forest para
   clasificar en espacios de embeddings densos y continuos, dado que
   estos espacios tienden a ser linealmente separables por clases.
4. Conservar los comentarios originales (sin filtrado agresivo de vocabulario)
   aporta información valiosa que los modelos de lenguaje pueden aprovechar.

### 5. Reflexión final del equipo

La actividad fortaleció los conocimientos en procesamiento de lenguaje natural (NLP), especialmente en embeddings densos y contextuales, modelos preentrenados de Hugging Face, tokenización, clasificación supervisada y evaluación de modelos (accuracy, precision, recall, F1-score y matrices de confusión).
Se concluye que los embeddings semánticos contextuales generados por modelos Transformer son una herramienta fundamental en IA aplicada al lenguaje. A diferencia de los métodos tradicionales (bag-of-words), que fallan ante sarcasmo o negaciones, los sentence embeddings condensan el contexto completo de las oraciones, capturando relaciones semánticas profundas y logrando representaciones mucho más ricas.
En la práctica, permitieron que una simple Regresión Logística superara a modelos más complejos como Random Forest, gracias a un espacio de características más eficiente y menos propenso al sobreajuste. Los embeddings contextuales del Transformer mostraron consistentemente mejor desempeño que los promedios de palabras individuales, especialmente en análisis de sentimientos.
En síntesis, los embeddings modernos son esenciales para aplicaciones avanzadas de NLP como análisis de sentimientos, búsqueda semántica, RAG y LLMs.


# **Fin de la Actividad de Vectores Embebidos - HuggingFace**